# FANet Phase-7B — Detached Soft-OR Gate: Smoke Test & Multi-Seed

**Hypothesis:** `MixPool(gating=soft_or, detach_feedback=True)` + Tversky loss phá vỡ Feedback Trap → FP giảm như M01/TD nhưng vòng lặp Feedback vẫn ACTIVE.

| Cell | Feedback | Loss | Gating | Detach | Label |
|---|---|---|---|---|---|
| M11 (TC, Phase 6) | ON | Tversky | binary | False | Feedback Trap (baseline) |
| **M12** | ON | Tversky | **soft_or** | **True** | **Feedback Fixed (Phase-7B)** |

**Notebook structure:**
- **Part 1** — Smoke Test M12, seed 42, 200ep (Fail-Fast @ ep40)
- **Part 2** — Inline Diagnostics: BN Drift, CosSim, Saturation vs M11 baseline
- **Part 3** — Multi-seed loop M11 vs M12 (5 seeds) — **GUARDED**, bỏ guard sau Part 1+2 OK

**Input datasets cần add trên Kaggle:**
1. `kvasir-sessile` (hoặc bất kỳ dataset nào có folder `images/` + `masks/`)
2. Phase 5 checkpoints dataset (có `ckpt_T0N.pth`) — cho Part 2 diagnostics
3. Phase 6 checkpoints dataset (có `ckpt_TC.pth`, `ckpt_TD.pth`) — cho Part 2 diagnostics

## 1. Setup

In [ ]:
!pip install -q albumentations opencv-python-headless tqdm scikit-learn

In [ ]:
import os, sys, time, random, json, math
import numpy as np
import cv2
import albumentations as A
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.utils import shuffle as sk_shuffle

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    !nvidia-smi | head -12

## 2. Dataset path

In [ ]:
# Auto-detect dataset path (same logic as Phase 5/6 notebooks)
CANDIDATES = [
    "/kaggle/input/kvasir-sessile",
    "/kaggle/input/kvasir-seg",
    "/kaggle/input/datasets/namnguynnnn/kvasir/kvasir-sessile/sessile-main-Kvasir-SEG",
]
SRC = None
for p in CANDIDATES:
    if os.path.isdir(p) and os.path.isdir(f"{p}/images"):
        SRC = p
        break
if SRC is None:
    for root, dirs, _ in os.walk("/kaggle/input"):
        if "images" in dirs and "masks" in dirs:
            SRC = root
            break
assert SRC is not None, "Kvasir dataset not found in /kaggle/input"
print(f"Found dataset: {SRC}")

DATASET_PATH = "/kaggle/working/Kvasir-SEG"
if not os.path.exists(DATASET_PATH):
    import shutil
    shutil.copytree(SRC, DATASET_PATH)

n_img = len([f for f in os.listdir(f"{DATASET_PATH}/images") if f.endswith('.jpg') or f.endswith('.png')])
print(f"Dataset ready: {n_img} images at {DATASET_PATH}")

## 3. Model blocks (Phase-7B MixPool)

In [ ]:
# ── SELayer ──────────────────────────────────────────────────────────────────
class SELayer(nn.Module):
    def __init__(self, channel, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False), nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False), nn.Sigmoid())
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        return x * self.fc(y).view(b, c, 1, 1).expand_as(x)

# ── ResidualBlock ─────────────────────────────────────────────────────────────
class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1); self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1); self.bn2 = nn.BatchNorm2d(out_c)
        self.conv3 = nn.Conv2d(in_c, out_c, 1, padding=0); self.bn3 = nn.BatchNorm2d(out_c)
        self.se = SELayer(out_c, out_c); self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        x1 = self.relu(self.bn1(self.conv1(x)))
        x2 = self.bn2(self.conv2(x1))
        x3 = self.se(self.bn3(self.conv3(x)))
        return self.relu(x2 + x3)

# ── MixPool (Phase-7B) ────────────────────────────────────────────────────────
class MixPool(nn.Module):
    """
    gating_mode:
        'hard'     - original FANet (binary threshold, zero gradient)
        'soft_or'  - smooth probabilistic OR: 1-(1-fmask)(1-m_fg)  [Phase-7B]
    detach_feedback:
        False - original: m_fg in gradient graph (Feedback Trap)
        True  - m_fg.detach() -> severs BN-drift + gradient attenuation [Phase-7B FIX]
    """
    def __init__(self, in_c, out_c, gating_mode='hard', detach_feedback=False):
        super().__init__()
        self.gating_mode = gating_mode
        self.detach_feedback = detach_feedback
        self.fmask = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, 1, 1), nn.Sigmoid())
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_c, out_c//2, 3, padding=1), nn.BatchNorm2d(out_c//2), nn.ReLU(inplace=True))
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_c, out_c//2, 3, padding=1), nn.BatchNorm2d(out_c//2), nn.ReLU(inplace=True))

    def forward(self, x, m):
        fmask = self.fmask(x)  # [B,1,h,w] fully differentiable
        sh, sw = m.shape[2] // x.shape[2], m.shape[3] // x.shape[3]
        m_fg = nn.MaxPool2d((sh, sw))(m)[:, 0:1]

        if self.detach_feedback:
            m_fg = m_fg.detach()  # KEY: severs Feedback Trap gradient

        if self.gating_mode == 'soft_or':
            keep = 1.0 - (1.0 - fmask) * (1.0 - m_fg)  # smooth prob-OR
        else:  # 'hard' — original FANet
            keep = torch.maximum((fmask > 0.5).float(), m_fg)

        x1 = self.conv1(x * keep)
        x2 = self.conv2(x)
        return torch.cat([x1, x2], dim=1)

# ── Encoder / Decoder blocks ─────────────────────────────────────────────────
class EncoderBlock(nn.Module):
    def __init__(self, in_c, out_c, gating_mode='hard', detach_feedback=False):
        super().__init__()
        self.r1 = ResidualBlock(in_c, out_c); self.r2 = ResidualBlock(out_c, out_c)
        self.p1 = MixPool(out_c, out_c, gating_mode=gating_mode, detach_feedback=detach_feedback)
        self.pool = nn.MaxPool2d((2, 2))
    def forward(self, x, m):
        x = self.r2(self.r1(x))
        p = self.p1(x, m)
        return self.pool(p), x

class DecoderBlock(nn.Module):
    def __init__(self, in_c, out_c, gating_mode='hard', detach_feedback=False):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_c, in_c, 4, stride=2, padding=1)
        self.r1 = ResidualBlock(in_c*2, out_c); self.r2 = ResidualBlock(out_c, out_c)
        self.p1 = MixPool(out_c, out_c, gating_mode=gating_mode, detach_feedback=detach_feedback)
    def forward(self, x, skip, m):
        x = self.r2(self.r1(torch.cat([self.up(x), skip], dim=1)))
        return self.p1(x, m)

# ── FANet ─────────────────────────────────────────────────────────────────────
class FANet(nn.Module):
    def __init__(self, gating_mode='hard', detach_feedback=False):
        super().__init__()
        kw = dict(gating_mode=gating_mode, detach_feedback=detach_feedback)
        self.e1 = EncoderBlock(3,   32,  **kw); self.e2 = EncoderBlock(32,  64,  **kw)
        self.e3 = EncoderBlock(64,  128, **kw); self.e4 = EncoderBlock(128, 256, **kw)
        self.d1 = DecoderBlock(256, 128, **kw); self.d2 = DecoderBlock(128, 64,  **kw)
        self.d3 = DecoderBlock(64,  32,  **kw); self.d4 = DecoderBlock(32,  16,  **kw)
        self.output = nn.Conv2d(17, 1, 1)
    def forward(self, x):
        inp, m = x[0], x[1]
        p1,s1 = self.e1(inp,m); p2,s2 = self.e2(p1,m)
        p3,s3 = self.e3(p2,m); p4,s4 = self.e4(p3,m)
        d1 = self.d1(p4,s4,m); d2 = self.d2(d1,s3,m)
        d3 = self.d3(d2,s2,m); d4 = self.d4(d3,s1,m)
        return self.output(torch.cat([d4, m[:, 0:1]], dim=1))

print("Model blocks defined.")
# Sanity check
_x = torch.randn(1, 3, 256, 256); _m = torch.zeros(1, 1, 256, 256)
_m12 = FANet(gating_mode='soft_or', detach_feedback=True)
assert _m12([_x, _m]).shape == (1, 1, 256, 256)
print(f"FANet (soft_or+detach) output: {_m12([_x, _m]).shape} OK")
del _x, _m, _m12

## 4. Dataset & utils (same split as Phase 5/6)

In [ ]:
# ── RLE helpers ───────────────────────────────────────────────────────────────
def rle_encode(mask):
    pixels = mask.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return list(runs)

def rle_decode(rle, shape):
    s = rle; mask = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for i in range(0, len(s), 2):
        start, length = int(s[i]) - 1, int(s[i+1])
        mask[start:start+length] = 1
    return mask.reshape(shape)

def rle_batch_to_tensor(mask_list, start, b, size):
    out = []
    for i in range(start, min(start+b, len(mask_list))):
        m = rle_decode(mask_list[i], size) if mask_list[i] else np.zeros(size, dtype=np.uint8)
        out.append(m)
    while len(out) < b: out.append(np.zeros(size, dtype=np.uint8))
    t = torch.from_numpy(np.stack(out, 0)).float().unsqueeze(1)  # [B,1,H,W]
    return t

def init_mask(paths, size):
    return [[] for _ in paths]

def seeding(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

# ── Dataset split (same 80/20 as Phase 5/6) ──────────────────────────────────
def load_data(path):
    imgs  = sorted([os.path.join(path, 'images', f)
                    for f in os.listdir(os.path.join(path, 'images'))
                    if f.lower().endswith(('.jpg','.png','.jpeg'))])
    masks = sorted([os.path.join(path, 'masks', f)
                    for f in os.listdir(os.path.join(path, 'masks'))
                    if f.lower().endswith(('.jpg','.png','.jpeg'))])
    split = int(0.8 * len(imgs))
    return (imgs[:split], masks[:split]), (imgs[split:], masks[split:])

# ── Albumentations Dataset ────────────────────────────────────────────────────
class KvasirDataset(Dataset):
    def __init__(self, images, masks, size, transform=None):
        self.images = images; self.masks = masks
        self.size = size; self.transform = transform
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img  = cv2.imread(self.images[idx], cv2.IMREAD_COLOR)
        img  = cv2.resize(img, self.size)
        mask = cv2.imread(self.masks[idx], cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, self.size)
        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']
        img  = torch.from_numpy(img.transpose(2,0,1)).float() / 255.0
        mask = torch.from_numpy(mask).float().unsqueeze(0) / 255.0
        return img, mask

# ── Binary metrics ────────────────────────────────────────────────────────────
def compute_bin_metrics(pred_prob, gt_np):
    pb = (pred_prob > 0.5).astype(np.uint8)
    gb = (gt_np > 0.5).astype(np.uint8)
    tp = (pb & gb).sum(); pp = pb.sum(); gp = gb.sum()
    fp = (pb & (1-gb)).sum()
    return {
        'dice': 2*tp/(pp+gp+1e-15),
        'prec': tp/(pp+1e-15),
        'rec':  tp/(gp+1e-15),
        'fpr':  fp/(gb.size+1e-15)
    }

print("Dataset & utils defined.")

## 5. Loss functions

In [ ]:
class DiceBCELoss(nn.Module):
    def forward(self, pred, gt):
        bce = F.binary_cross_entropy_with_logits(pred, gt)
        pred_s = torch.sigmoid(pred)
        inter = (pred_s * gt).sum(dim=(2,3))
        dice = 1 - (2*inter + 1) / (pred_s.sum(dim=(2,3)) + gt.sum(dim=(2,3)) + 1)
        return bce + dice.mean()

class TverskyAsymLoss(nn.Module):
    """0.5 * DiceBCE + 0.5 * Tversky(alpha, beta) — same as Phase 6 TC/TD."""
    def __init__(self, alpha=0.7, beta=0.3):
        super().__init__()
        self.alpha = alpha; self.beta = beta
        self.base = DiceBCELoss()
    def forward(self, pred, gt):
        base_loss = self.base(pred, gt)
        p = torch.sigmoid(pred)
        tp  = (p * gt).sum(dim=(2,3))
        fp  = (p * (1-gt)).sum(dim=(2,3))
        fn  = ((1-p) * gt).sum(dim=(2,3))
        tversky = (tp + 1) / (tp + self.alpha*fp + self.beta*fn + 1)
        return 0.5 * base_loss + 0.5 * (1 - tversky.mean())

print("Loss functions defined.")

## 6. Train/Evaluate functions

In [ ]:
def train_one_epoch(model, loader, mask_rle, optimizer, loss_fn, size, no_feedback=False):
    model.train(); total = 0; new_mask = []
    for i, (x, y) in enumerate(loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        b = y.shape[0]
        m = (torch.zeros(b,1,*size).to(DEVICE) if no_feedback
             else rle_batch_to_tensor(mask_rle, i*b, b, size).to(DEVICE))
        optimizer.zero_grad()
        loss = loss_fn(model([x, m]), y); loss.backward(); optimizer.step()
        total += loss.item()
        with torch.no_grad():
            pred = torch.sigmoid(model([x, m])).cpu().numpy()
            for py in pred:
                new_mask.append(rle_encode((np.squeeze(py) > 0.5).astype(np.uint8)))
    return total / len(loader), new_mask

def eval_one_epoch(model, loader, mask_rle, loss_fn, size, no_feedback=False):
    model.eval(); total = 0; new_mask = []
    agg = {'dice':0,'prec':0,'rec':0,'fpr':0}
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            x, y = x.to(DEVICE), y.to(DEVICE)
            b = y.shape[0]
            m = (torch.zeros(b,1,*size).to(DEVICE) if no_feedback
                 else rle_batch_to_tensor(mask_rle, i*b, b, size).to(DEVICE))
            pred_logit = model([x, m])
            total += loss_fn(pred_logit, y).item()
            pred = torch.sigmoid(pred_logit).cpu().numpy()
            gt   = y.cpu().numpy()
            for py, gy in zip(pred, gt):
                new_mask.append(rle_encode((np.squeeze(py) > 0.5).astype(np.uint8)))
                bm = compute_bin_metrics(np.squeeze(py), np.squeeze(gy))
                for k in agg: agg[k] += bm[k]
    n = len(loader.dataset)
    return total/len(loader), new_mask, {k: v/n for k,v in agg.items()}

print("Train/eval functions defined.")

---
## Part 1 — Smoke Test: M12 (soft_or + detach + Tversky), Seed 42

**Fail-Fast @ epoch 40:** `bin_dice` phải > 0.05. Nếu collapse giống TB Phase 5 (dice=0.008) → dừng ngay.

**Expected runtime:** ~18-20 phút/200ep trên Kaggle T4.

In [ ]:
# ════════════════════════════════════════════
# PART 1: SMOKE TEST — M12
# ════════════════════════════════════════════
SMOKE_SEED    = 42
SMOKE_EPOCHS  = 200
FAIL_FAST_EP  = 40        # check collapse here
SIZE          = (256, 256)
BATCH         = 2
LR            = 1e-4

CKPT_DIR = "/kaggle/working/checkpoints_phase7b"
LOG_DIR  = "/kaggle/working/logs_phase7b"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR,  exist_ok=True)

M12_CKPT = f"{CKPT_DIR}/ckpt_M12_seed{SMOKE_SEED}.pth"
M12_LOG  = f"{LOG_DIR}/M12_seed{SMOKE_SEED}.csv"

seeding(SMOKE_SEED)
(tr_x, tr_y), (vl_x, vl_y) = load_data(DATASET_PATH)
tr_x, tr_y = sk_shuffle(tr_x, tr_y, random_state=SMOKE_SEED)
print(f"Train={len(tr_x)}, Val={len(vl_x)}")

aug = A.Compose([
    A.Rotate(limit=35, p=0.3), A.HorizontalFlip(p=0.3), A.VerticalFlip(p=0.3),
    A.CoarseDropout(p=0.3, num_holes=10, hole_height=32, hole_width=32),
])
tr_loader = DataLoader(KvasirDataset(tr_x, tr_y, SIZE, aug),  batch_size=BATCH, shuffle=False, num_workers=2)
vl_loader = DataLoader(KvasirDataset(vl_x, vl_y, SIZE, None), batch_size=BATCH, shuffle=False, num_workers=2)

model_m12 = FANet(gating_mode='soft_or', detach_feedback=True).to(DEVICE)
print(f"M12 params: {sum(p.numel() for p in model_m12.parameters()):,}")

loss_fn   = TverskyAsymLoss(alpha=0.7, beta=0.3)
optimizer = torch.optim.Adam(model_m12.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)

tr_mask = init_mask(tr_x, SIZE)
vl_mask = init_mask(vl_x, SIZE)
best    = float('inf')

# Write CSV header
with open(M12_LOG, 'w') as f:
    f.write("epoch,tr_loss,vl_loss,bin_dice,bin_fpr,bin_prec,bin_rec\n")

print(f"\nStarting M12 smoke test | seed={SMOKE_SEED} | gating=soft_or | detach=True | loss=Tversky(0.7/0.3)")
print("=" * 80)

for ep in range(SMOKE_EPOCHS):
    t0 = time.time()
    tl, tr_mask_new = train_one_epoch(model_m12, tr_loader, tr_mask, optimizer, loss_fn, SIZE)
    vl, vl_mask_new, bm = eval_one_epoch(model_m12, vl_loader, vl_mask, loss_fn, SIZE)
    scheduler.step(vl)

    if vl < best:
        best = vl
        torch.save(model_m12.state_dict(), M12_CKPT)
        tr_mask, vl_mask = tr_mask_new, vl_mask_new

    elapsed = time.time() - t0
    row = (f"Ep {ep+1:03}/{SMOKE_EPOCHS} | {elapsed:.0f}s | "
           f"TrLoss={tl:.4f} VlLoss={vl:.4f} | "
           f"Dice={bm['dice']:.4f} FPR={bm['fpr']:.4f} [best={best:.4f}]")
    print(row)
    with open(M12_LOG, 'a') as f:
        f.write(f"{ep+1},{tl:.4f},{vl:.4f},{bm['dice']:.4f},{bm['fpr']:.4f},{bm['prec']:.4f},{bm['rec']:.4f}\n")

    # ── Fail-Fast @ epoch 40 ──────────────────────────────────────
    if (ep + 1) == FAIL_FAST_EP:
        print(f"\n{'='*55}")
        print(f"FAIL-FAST CHECK @ Epoch {FAIL_FAST_EP}")
        print(f"  bin_dice : {bm['dice']:.4f}  (threshold > 0.05)")
        print(f"  bin_fpr  : {bm['fpr']:.4f}")
        if bm['dice'] < 0.05:
            raise RuntimeError(f"FAIL-FAST: dice={bm['dice']:.4f} < 0.05 @ ep{FAIL_FAST_EP}. Model collapsed!")
        print("  [PASS] Continuing to full run.")
        print(f"{'='*55}\n")

print(f"\nSmoke Test DONE. Best val_loss={best:.4f}. Checkpoint: {M12_CKPT}")

---
## Part 2 — Inline Diagnostics: M12 vs M11 (Feedback Trap baseline)

Load M11 (TC, Phase 6) từ input dataset và so sánh trực tiếp với M12 vừa train.

> **Cần add input:** Phase 6 checkpoint dataset chứa `ckpt_TC.pth`.

In [ ]:
# ════════════════════════════════════════════
# PART 2: DIAGNOSTICS — M12 vs M11
# ════════════════════════════════════════════

# Auto-detect Phase 6 checkpoint (TC = M11 = Feedback Trap)
M11_CKPT_PATH = None
for root, dirs, files in os.walk("/kaggle/input"):
    for fname in files:
        if "TC" in fname and fname.endswith(".pth"):
            M11_CKPT_PATH = os.path.join(root, fname)
            break
    if M11_CKPT_PATH: break

print(f"M11 (TC) checkpoint: {M11_CKPT_PATH}")
if M11_CKPT_PATH is None:
    print("[WARN] M11 checkpoint not found — skipping M11 comparison. Only M12 metrics will be shown.")

# ── Load models ───────────────────────────────────────────────────
models_diag = {"M12": (FANet(gating_mode='soft_or', detach_feedback=True).to(DEVICE), M12_CKPT)}
if M11_CKPT_PATH:
    models_diag["M11"] = (FANet(gating_mode='hard', detach_feedback=False).to(DEVICE), M11_CKPT_PATH)

for key, (net, ckpt_p) in models_diag.items():
    net.load_state_dict(torch.load(ckpt_p, map_location=DEVICE))
    net.eval()
    print(f"[OK] {key} loaded from {ckpt_p}")

# ── BN stats helper ───────────────────────────────────────────────
def get_bn_stats(model):
    return {n: {"mean": m.running_mean.clone().cpu().numpy(),
                "var":  m.running_var.clone().cpu().numpy()}
            for n, m in model.named_modules() if isinstance(m, nn.BatchNorm2d)}

def kl_div(m1, v1, m0, v0, eps=1e-5):
    v0, v1 = np.maximum(v0, eps), np.maximum(v1, eps)
    return float(np.mean(np.log(v0**.5/v1**.5) + (v1+(m1-m0)**2)/(2*v0) - 0.5))

bn_all = {k: get_bn_stats(net) for k, (net, _) in models_diag.items()}

# ── Per-image diagnostics ─────────────────────────────────────────
feat_store = {}
hooks = {}
for k, (net, _) in models_diag.items():
    def make_hook(key):
        def fn(m, i, o):
            feat_store[key] = (o[1] if isinstance(o, (tuple,list)) else o).detach()
        return fn
    hooks[k] = net.e4.register_forward_hook(make_hook(k))

diag = {k: {"cos_sim":[], "sat":[], "ent":[], "gnorm":[]} for k in models_diag}
vl_diag = DataLoader(KvasirDataset(vl_x, vl_y, SIZE, None), batch_size=1, shuffle=False)

for x_b, y_b in vl_diag:
    x_b = x_b.to(DEVICE)
    y_np = y_b[0, 0].numpy()
    if y_np.max() > 0:
        dist = cv2.distanceTransform((1-(y_np>0.5).astype(np.uint8)), cv2.DIST_L2, 3)
        bnd_np = (dist>0) & (dist<=20)
        far_np = dist > 30
    else:
        bnd_np = far_np = None

    for k, (net, _) in models_diag.items():
        m_in = torch.zeros(1, 1, *SIZE, device=DEVICE, requires_grad=True)
        out = net([x_b, m_in])
        feat = feat_store.get(k)
        if feat is not None and bnd_np is not None:
            h, w = feat.shape[2], feat.shape[3]
            bs = cv2.resize(bnd_np.astype(np.uint8),(w,h),interpolation=cv2.INTER_NEAREST).astype(bool)
            fs = cv2.resize(far_np.astype(np.uint8),(w,h),interpolation=cv2.INTER_NEAREST).astype(bool)
            fhw = feat[0].permute(1,2,0).cpu().numpy()
            if bs.sum()>0 and fs.sum()>0:
                vb = fhw[bs].mean(0); vf = fhw[fs].mean(0)
                cos = float(np.dot(vb,vf)/(np.linalg.norm(vb)*np.linalg.norm(vf)+1e-8))
                diag[k]["cos_sim"].append(cos)
        prob = torch.sigmoid(out).detach()
        diag[k]["sat"].append(float(((prob<0.01)|(prob>0.99)).float().mean()))
        diag[k]["ent"].append(float(-(prob*torch.log(prob+1e-8)+(1-prob)*torch.log(1-prob+1e-8)).mean()))
        out.sum().backward()
        diag[k]["gnorm"].append(float(m_in.grad.norm(2)) if m_in.grad is not None else 0.0)
        net.zero_grad()

for k in models_diag: hooks[k].remove()

# ── Print summary ─────────────────────────────────────────────────
print("\n" + "="*70)
print("DIAGNOSTIC SUMMARY")
print("="*70)
print(f"{'Model':<8} {'CosSim':>8} {'Sat%':>8} {'Entropy':>9} {'GradNorm':>12}")
print("-"*50)

summary_diag = {}
for k in models_diag:
    d = diag[k]
    cos = np.mean(d["cos_sim"]) if d["cos_sim"] else float("nan")
    sat = np.mean(d["sat"]) * 100
    ent = np.mean(d["ent"])
    gn  = np.mean(d["gnorm"])
    summary_diag[k] = {"cos_sim": round(cos,4), "sat_pct": round(sat,2),
                       "entropy": round(ent,4), "grad_norm": round(gn,4)}
    print(f"{k:<8} {cos:>8.4f} {sat:>7.2f}% {ent:>9.4f} {gn:>12.4f}")

# ── Decision ──────────────────────────────────────────────────────
print("\n" + "="*70)
print("DECISION")
if "M11" in summary_diag and "M12" in summary_diag:
    sat_d = summary_diag["M11"]["sat_pct"]  - summary_diag["M12"]["sat_pct"]
    gn_d  = summary_diag["M12"]["grad_norm"] - summary_diag["M11"]["grad_norm"]
    print(f"  Saturation M11-M12 : {sat_d:+.2f}pp  (want > 0 = M12 less saturated)")
    print(f"  Grad Norm  M12-M11 : {gn_d:+.4f}    (want > 0 = M12 more gradient)")
    if sat_d > 0 and gn_d > 0:
        print("  [SUCCESS] Feedback Trap appears severed. Proceed to Part 3 multi-seed.")
    else:
        print("  [CHECK] Some metrics unexpected. Review loss curves before multi-seed.")
else:
    print(f"  M12 metrics: {summary_diag.get('M12', 'N/A')}")
    print("  (Add Phase 6 checkpoint dataset to enable M11 comparison)")

# Save JSON
DIAG_OUT = f"/kaggle/working/logs_phase7b/diagnostics_phase7b.json"
with open(DIAG_OUT, 'w') as f:
    json.dump(summary_diag, f, indent=2)
print(f"\nSaved: {DIAG_OUT}")

---
## Part 3 — Multi-Seed Loop: M11 vs M12 (5 seeds)

> **INSTRUCTION:** Xoá dòng `raise RuntimeError(...)` bên dưới sau khi Part 1 và Part 2 thành công.

Seeds: `[42, 1337, 2024, 7, 99]` — seed 42 đã có từ Part 1, sẽ được skip tự động.

GPU time estimate: ~20 phút/cell/seed → tổng ~**3.3h** cho 2 cells × 5 seeds trên T4.

In [ ]:
# ════════════════════════════════════════════
# PART 3: MULTI-SEED — M11 vs M12
# Remove the guard below after Part 1+2 PASS
# ════════════════════════════════════════════
# (Guard commented out) raise RuntimeError(
#     "GUARD: Delete this line after Part 1 & Part 2 succeed to run multi-seed."
# )

SEEDS  = [42, 1337, 2024, 7, 99]
EPOCHS = 200

CELL_CFGS = {
    "M11": dict(gating_mode='hard',    detach_feedback=False, no_feedback=False),
    "M12": dict(gating_mode='soft_or', detach_feedback=True,  no_feedback=False),
}

all_res = {}
for seed in SEEDS:
    for cell, cfg in CELL_CFGS.items():
        run_id   = f"{cell}_seed{seed}"
        ckpt_p   = f"{CKPT_DIR}/ckpt_{cell}_seed{seed}.pth"
        log_p    = f"{LOG_DIR}/{run_id}.csv"

        if os.path.exists(ckpt_p):
            print(f"[SKIP] {run_id} already done."); continue

        print(f"\n{'='*55}\nTraining {run_id}\n{'='*55}")
        seeding(seed)
        (tr_x2, tr_y2), (vl_x2, vl_y2) = load_data(DATASET_PATH)
        tr_x2, tr_y2 = sk_shuffle(tr_x2, tr_y2, random_state=seed)

        tr_l = DataLoader(KvasirDataset(tr_x2, tr_y2, SIZE, aug),  batch_size=BATCH, shuffle=False, num_workers=2)
        vl_l = DataLoader(KvasirDataset(vl_x2, vl_y2, SIZE, None), batch_size=BATCH, shuffle=False, num_workers=2)

        net     = FANet(gating_mode=cfg['gating_mode'], detach_feedback=cfg['detach_feedback']).to(DEVICE)
        lfn     = TverskyAsymLoss(0.7, 0.3)
        opt     = torch.optim.Adam(net.parameters(), lr=LR)
        sched   = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', patience=5)
        tr_m    = init_mask(tr_x2, SIZE)
        vl_m    = init_mask(vl_x2, SIZE)
        best    = float('inf')
        no_fb   = cfg['no_feedback']

        with open(log_p, 'w') as f: f.write("epoch,tr_loss,vl_loss,dice,fpr,prec,rec\n")

        for ep in range(EPOCHS):
            t0 = time.time()
            tl, tr_m_new = train_one_epoch(net, tr_l, tr_m, opt, lfn, SIZE, no_fb)
            vl, vl_m_new, bm = eval_one_epoch(net, vl_l, vl_m, lfn, SIZE, no_fb)
            sched.step(vl)
            if vl < best:
                best = vl; torch.save(net.state_dict(), ckpt_p)
                tr_m, vl_m = tr_m_new, vl_m_new
            elapsed = time.time()-t0
            row = f"Ep {ep+1:03}/{EPOCHS}|{elapsed:.0f}s|TrL={tl:.4f} VlL={vl:.4f}|Dice={bm['dice']:.4f} FPR={bm['fpr']:.4f}"
            print(row)
            with open(log_p, 'a') as f:
                f.write(f"{ep+1},{tl:.4f},{vl:.4f},{bm['dice']:.4f},{bm['fpr']:.4f},{bm['prec']:.4f},{bm['rec']:.4f}\n")

        all_res[run_id] = {"cell": cell, "seed": seed, "best_vl": round(best,4), **{k: round(v,4) for k,v in bm.items()}}
        print(f"[DONE] {run_id} best={best:.4f}")

# Save & print aggregate
with open(f"{LOG_DIR}/multiseed_summary.json", 'w') as f:
    json.dump(all_res, f, indent=2)

print(f"\n{'RunID':<25} {'Dice':>8} {'FPR':>8} {'Prec':>8} {'Rec':>8}")
print("-"*62)
for rid, r in all_res.items():
    print(f"{rid:<25} {r['dice']:>8.4f} {r['fpr']:>8.4f} {r['prec']:>8.4f} {r['rec']:>8.4f}")